# Install Required Packages

In [ ]:
# Install the Libraries Needed for the Project
# - paho-mqtt     : Helps devices send and receive MQTT messages.
# - streamlit     : Builds a simple web application.
# - plotly        : Creates charts and graphs.
# - python-dotenv : Reads settings from a .env file.
# - requests      : Connects to web APIs and websites.

!pip install paho-mqtt streamlit plotly python-dotenv requests --quiet


# Create the Environment File (.env)

In [ ]:
# Create the Environment Variables File
# This file stores the project settings in one place.

# - FIREBASE_URL  : The link to the Firebase database.
# - FIREBASE_AUTH : The key used to access the database.
# - MQTT_BROKER   : The MQTT server used to send and receive data.
# - MQTT_PORT     : The port used to connect to the MQTT server.
# - MQTT_TOPIC    : The topic where the farm data is shared.

%%writefile .env
FIREBASE_URL="https://smartfarmiot-76187-default-rtdb.firebaseio.com"
FIREBASE_AUTH="LCIE3zkzD3ZSscEhXYGM0os3pDoK51HesZo218xE"
MQTT_BROKER="broker.hivemq.com"
MQTT_PORT=1883
MQTT_TOPIC="Group6_SmartFarm"

Overwriting .env


# Write the MQTT Bridge Script (mqtt_bridge.py)

In [ ]:
%%writefile mqtt_bridge.py

# Import the libraries used in the program
import json
import time
import logging
import requests
from datetime import datetime, timezone
from dotenv import load_dotenv
import os
import paho.mqtt.client as mqtt

# Load the settings stored in the .env file
load_dotenv()

# MQTT connection details
MQTT_BROKER = os.getenv("MQTT_BROKER", "broker.hivemq.com")
MQTT_PORT = int(os.getenv("MQTT_PORT", 1883))
MQTT_TOPIC = os.getenv("MQTT_TOPIC", "Group6_SmartFarm")
MQTT_CLIENT = "smartfarm-bridge"

# Firebase connection details
FIREBASE_URL = os.getenv("FIREBASE_URL")
FIREBASE_AUTH = os.getenv("FIREBASE_AUTH")

# Set up logging to display program messages
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("smartfarm-bridge")

# Add new data to Firebase
def firebase_post(endpoint: str, payload: dict) -> bool:
    url = f"{FIREBASE_URL}/{endpoint}.json?auth={FIREBASE_AUTH}"
    try:
        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()
        return True
    except requests.RequestException as e:
        log.error(f"Firebase POST failed ({endpoint}): {e}")
        return False

# Update data in Firebase
def firebase_put(endpoint: str, payload: dict) -> bool:
    url = f"{FIREBASE_URL}/{endpoint}.json?auth={FIREBASE_AUTH}"
    try:
        r = requests.put(url, json=payload, timeout=10)
        r.raise_for_status()
        return True
    except requests.RequestException as e:
        log.error(f"Firebase PUT failed ({endpoint}): {e}")
        return False

# Save the latest sensor data
def write_to_firebase(data: dict) -> None:
    now = datetime.now(timezone.utc).isoformat()
    data["timestamp"] = now

    ok_history = firebase_post("readings", data)
    if ok_history:
        log.info("Firebase /readings — new record written.")

    ok_latest = firebase_put("latest", data)
    if ok_latest:
        log.info("Firebase /latest — updated.")

    device_payload = {
        "status": "online",
        "last_seen": now,
        "mqtt_active": True,
        "ip_address": "via-mqtt"
    }
    firebase_put("device", device_payload)

# Run when the MQTT client connects
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        log.info(f"Connected to MQTT broker: {MQTT_BROKER}:{MQTT_PORT}")
        client.subscribe(MQTT_TOPIC)
    else:
        log.error(f"MQTT connection failed — return code {rc}")

# Run when the MQTT client disconnects
def on_disconnect(client, userdata, rc):
    log.warning(f"MQTT disconnected (rc={rc}). Will auto-reconnect.")
    firebase_put("device", {
        "status": "offline",
        "last_seen": datetime.now(timezone.utc).isoformat(),
        "mqtt_active": False,
        "ip_address": "unknown"
    })

# Process each MQTT message
def on_message(client, userdata, msg):
    try:
        raw = msg.payload.decode("utf-8")
        log.info(f"MQTT message received on [{msg.topic}]: {raw}")
        data = json.loads(raw)
        validate_and_write(data)
    except Exception as e:
        log.error(f"Unexpected error processing message: {e}")

# Sensor values that must be included
REQUIRED_FIELDS = {"soil_moisture", "temperature", "humidity", "light_lux"}

# Check the data before saving it
def validate_and_write(data: dict) -> None:
    missing = REQUIRED_FIELDS - data.keys()
    if missing:
        log.warning(f"Payload missing fields: {missing} — skipping write.")
        return

    # Create alerts when values go beyond the limits
    data["alerts"] = {
        "irrigate": data["soil_moisture"] < 30.0,
        "heat_stress": data["temperature"] > 35.0,
        "dry_air": data["humidity"] < 40.0,
        "poor_light": data["light_lux"] < 200.0
    }

    write_to_firebase(data)

# Start the MQTT bridge
def main():
    if not FIREBASE_URL or not FIREBASE_AUTH:
        log.critical("FIREBASE_URL and FIREBASE_AUTH must be set in .env — aborting.")
        return

    client = mqtt.Client(client_id=MQTT_CLIENT, clean_session=True)
    client.on_connect = on_connect
    client.on_disconnect = on_disconnect
    client.on_message = on_message

    client.reconnect_delay_set(min_delay=2, max_delay=30)
    client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)

    try:
        client.loop_forever()
    except KeyboardInterrupt:
        client.disconnect()

# Run the program
if __name__ == "__main__":
    main()

Overwriting mqtt_bridge.py


# Start the MQTT Bridge Background Process

In [ ]:
# Run the MQTT bridge in the background and save all logs to a file.
!python3 mqtt_bridge.py > bridge_output.log 2>&1 &

# Write the Streamlit Dashboard Code (app.py)

# Dependencies, Configuration & Custom CSS

In [ ]:
%%writefile app_config.py
import streamlit as st

def get_svg_icon(name: str, color: str = None) -> str:
    """Returns a clean, high-fidelity SVG icon string instead of basic emojis."""
    # Fallback to modern themed colors if none provided
    c = color if color else "currentColor"

    icons = {
        "soil": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 22c5.523 0 10-4.477 10-10S17.523 2 12 2 2 6.477 2 12s4.477 10 10 10z"/><path d="M12 6v12"/><path d="M8 10h8"/></svg>""",
        "temp": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M14 14.76V3.5a2.5 2.5 0 0 0-5 0v11.26a4.5 4.5 0 1 0 5 0z"/></svg>""",
        "humidity": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 22a7 7 0 0 0 7-7c0-4.3-7-13-7-13S5 10.7 5 15a7 7 0 0 0 7 7z"/></svg>""",
        "light": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="12" r="4"/><path d="M12 2v2"/><path d="M12 20v2"/><path d="M4.93 4.93l1.41 1.41"/><path d="M17.66 17.66l1.41 1.41"/><path d="M2 12h2"/><path d="M20 12h2"/><path d="M6.34 17.66l-1.41 1.41"/><path d="M19.07 4.93l-1.41 1.41"/></svg>""",
        "alert": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M10.29 3.86L1.82 18a2 2 0 0 0 1.71 3h16.94a2 2 0 0 0 1.71-3L13.71 3.86a2 2 0 0 0-3.42 0z"/><line x1="12" y1="9" x2="12" y2="13"/><line x1="12" y1="17" x2="12.01" y2="17"/></svg>""",
        "check": f"""<svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>"""
    }
    return icons.get(name, "")

def apply_custom_styles():
    """Applies clean CSS rules that shift automatically with your Light/Dark settings."""
    st.set_page_config(
        page_title="SmartFarm Precision IoT",
        page_icon="🌱",
        layout="wide",
        initial_sidebar_state="collapsed",
    )

    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@500;700&display=swap');

    /* Global Typography Reset */
    html, body, [class*="css"] {
        font-family: 'Plus Jakarta Sans', sans-serif;
    }

    /* ── DYNAMIC RADIANT BACKGROUND CANVAS ── */
    /* Light Mode Active Canvas */
    [data-theme="light"] .stApp {
        background: radial-gradient(circle at 50% 10%, #f4fcf0 0%, #e3f2dc 100%) !important;
        color: #1e3512 !important;
    }
    /* Dark Mode Active Canvas */
    [data-theme="dark"] .stApp {
        background: radial-gradient(circle at 50% 10%, #162d16 0%, #0a140a 100%) !important;
        color: #e5f5dd !important;
    }

    /* ── THEMED REGULAR CARDS ── */
    .custom-card {
        border-radius: 16px;
        padding: 24px;
        margin-bottom: 1rem;
        box-shadow: 0 8px 24px 0 rgba(0, 0, 0, 0.08);
        transition: all 0.3s ease;
    }
    [data-theme="light"] .custom-card {
        background: #ffffff;
        border: 1px solid #cfe6c3;
    }
    [data-theme="dark"] .custom-card {
        background: linear-gradient(145deg, #1b361b 0%, #112211 100%);
        border: 1px solid #2d592d;
    }

    .card-header-wrapper {
        display: flex;
        align-items: center;
        gap: 8px;
        margin-bottom: 12px;
    }
    .card-title-text {
        font-size: 0.75rem;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.12em;
    }
    [data-theme="light"] .card-title-text { color: #3b7a1e; }
    [data-theme="dark"] .card-title-text { color: #8ce65e; }

    /* Custom Header Component */
    .sf-header {
        border-radius: 12px;
        padding: 20px;
        margin-bottom: 1.5rem;
        display: flex;
        justify-content: space-between;
        align-items: center;
    }
    [data-theme="light"] .sf-header { background: #e8f5e1; border: 1px solid #cce6be; }
    [data-theme="dark"] .sf-header { background: #112411; border: 1px solid #234723; }

    /* ── STREAMLIT METRIC EXTENSION INJECTIONS ── */
    div[data-testid="stMetricValue"] {
        font-family: 'Space Grotesk', sans-serif !important;
        font-weight: 700 !important;
        font-size: 2.3rem !important;
    }
    [data-theme="light"] div[data-testid="stMetricValue"] { color: #1e3512 !important; }
    [data-theme="dark"] div[data-testid="stMetricValue"] { color: #ffffff !important; }

    [data-theme="light"] div[data-testid="stMetricLabel"] { color: #496e35 !important; }
    [data-theme="dark"] div[data-testid="stMetricLabel"] { color: #a4cfa4 !important; }

    /* Form Inputs Overrides */
    .stSelectbox > div > div {
        border-radius: 8px;
    }
    </style>
    """, unsafe_allow_html=True)

Overwriting app_config.py


# FIREBASE CONNECTION

In [ ]:
%%writefile app_engine.py
import streamlit as st
import pandas as pd
import requests as _requests
from datetime import datetime, timezone
import os
from dotenv import load_dotenv

# Load settings stored in the .env file
load_dotenv()

#  LIVE PATHWAY CONFIGURATION
DEMO_MODE     = False  # FALSE: Bypasses simulation logic to stream real IoT sensor metrics
FIREBASE_URL  = os.getenv("FIREBASE_URL", "https://smartfarmiot-76187-default-rtdb.firebaseio.com")
FIREBASE_AUTH = os.getenv("FIREBASE_AUTH", "")

def _fb_get(path: str) -> dict | None:
    """Performs extraction against specified database nodes."""
    url = f"{FIREBASE_URL}/{path}.json?auth={FIREBASE_AUTH}"
    try:
        r = _requests.get(url, timeout=8)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

def fetch_latest() -> dict:
    if DEMO_MODE: return _demo_latest()
    data = _fb_get("latest")
    return data if data else _demo_latest()

def fetch_history(hours: int = 6) -> list[dict]:
    if DEMO_MODE: return _demo_history(hours)
    data = _fb_get("readings")
    if not data: return _demo_history(hours)

    # Convert the dictionary payload from Firebase into a structured list
    records = list(data.values())
    try:
        records.sort(key=lambda r: r.get("timestamp", ""))
        cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=hours)
        return [r for r in records if pd.Timestamp(r["timestamp"]) >= cutoff]
    except Exception:
        return records

def fetch_device() -> dict | None:
    if DEMO_MODE:
        return {"status": "online", "last_seen": datetime.now(timezone.utc).isoformat(), "mqtt_active": True}
    data = _fb_get("device")
    return data if data else {"status": "offline", "last_seen": datetime.now(timezone.utc).isoformat(), "mqtt_active": False}


# FALLBACK (safety if database nodes are empty)


def _demo_latest() -> dict:
    return {
        "soil_moisture": 48.0, "temperature": 24.5, "humidity": 52.0, "light_lux": 450.0,
        "alerts": {"irrigate": False, "heat_stress": False, "dry_air": False, "poor_light": False},
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

def _demo_history(hours: int) -> list[dict]:
    import math
    from datetime import timedelta
    now = datetime.now(timezone.utc)
    records = []
    for i in range(hours * 6):
        ts = now - timedelta(minutes=10 * ((hours * 6) - i))
        records.append({
            "soil_moisture": round(45 + 5 * math.sin(i/5), 1),
            "temperature": round(22 + 3 * math.cos(i/8), 1),
            "humidity": round(50 + 8 * math.sin(i/10), 1),
            "light_lux": round(max(0, 500 + 300 * math.sin(i/6)), 1),
            "timestamp": ts.isoformat()
        })
    return records

Overwriting app_engine.py


# Reusable UI Elements, Chart Creation

In [ ]:
%%writefile app_ui_widgets.py
import streamlit as st
import plotly.graph_objects as go
import pandas as pd
from app_config import get_svg_icon

def make_combined_chart(df, is_dark_mode=True):
    """Generates an asset history chart that scales lines dynamically based on style themes."""
    axis_label_color = "#64748b"
    axis_line_color  = "#475569"
    grid_line_color  = "rgba(148, 163, 184, 0.15)"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["timestamp"], y=df["soil_moisture"],
        name="Soil Moisture (%)", line=dict(color="#2563eb", width=3),
        fill="tozeroy", fillcolor="rgba(37, 99, 235, 0.04)"
    ))
    fig.add_trace(go.Scatter(
        x=df["timestamp"], y=df["temperature"],
        name="Temp (°C)", line=dict(color="#ea580c", width=2.5),
    ))

    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=40, r=20, t=20, b=40),
        height=280,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="right",
            x=1,
            font=dict(color=axis_label_color, size=11)
        ),
        xaxis=dict(
            showgrid=False,
            tickfont=dict(color=axis_label_color, size=11, family="'Space Grotesk', sans-serif"),
            showline=True,
            linecolor=axis_line_color,
            linewidth=2,
            mirror=False
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=grid_line_color,
            tickfont=dict(color=axis_label_color, size=11, family="'Space Grotesk', sans-serif"),
            showline=True,
            linecolor=axis_line_color,
            linewidth=2,
            mirror=False
        )
    )
    return fig

def render_advisory_card(soil: float, temp: float, hum: float, light: float):
    """Presents a comprehensive farm health status checklist for all telemetry metrics."""

    # Threshold checks
    is_soil_flooded = soil >= 85.0
    is_soil_dry = soil < 30.0
    is_temp_stressed = temp > 35.0 or temp < 15.0
    is_hum_dry = hum < 40.0
    is_light_low = light < 200.0

    # Main Card style setup
    if is_soil_flooded or is_soil_dry or is_temp_stressed or is_hum_dry or is_light_low:
        border, header_color, icon_name, title = "#ef4444", "#ef4444", "alert", "Action Plan Required"
    else:
        border, header_color, icon_name, title = "#10b981", "#10b981", "check", "All Systems Operating Safely"

    # Status sentences
    if is_soil_flooded:
        soil_status = f"<b>Soil Moisture ({soil:.1f}%):</b> Saturated. Turn off water pumps now."
    elif is_soil_dry:
        soil_status = f" <b>Soil Moisture ({soil:.1f}%):</b> Dangerously dry! Turn on watering systems."
    else:
        soil_status = f" <b>Soil Moisture ({soil:.1f}%):</b> Healthy moisture level."

    if is_temp_stressed:
        temp_status = f" <b>Temperature ({temp:.1f}°C):</b> Too hot or too cold for stable plant growth."
    else:
        temp_status = f" <b>Temperature ({temp:.1f}°C):</b> Safe and comfortable temperature."

    if is_hum_dry:
        hum_status = f" <b>Air Humidity ({hum:.1f}%):</b> Air is too dry. Plants might dry out."
    else:
        hum_status = f" <b>Air Humidity ({hum:.1f}%):</b> Healthy humidity level."

    if is_light_low:
        light_status = f" <b>Light Level ({light:.1f} lx):</b> Low light.Plant growth may be affected."
    else:
        light_status = f" <b>Light Level ({light:.1f} lx):</b> Good sunlight for the crops."

    svg_icon = get_svg_icon(icon_name, header_color)

    st.markdown(
        f'<div class="custom-card" style="border-left: 5px solid {border}; display: flex; gap: 16px; align-items: flex-start; padding: 16px; background: rgba(0,0,0,0.01); border-radius: 4px;">'
        f'  <div style="margin-top: 4px;">{svg_icon}</div>'
        f'  <div style="width: 100%;">'
        f'    <div class="card-title-text" style="color: {header_color}; font-weight: 600; margin-bottom: 8px;">{title}</div>'
        f'    <div style="font-size: 0.9rem; line-height: 1.6; opacity: 0.9;">'
        f'       <div style="margin-bottom: 4px;">{soil_status}</div>'
        f'       <div style="margin-bottom: 4px;">{temp_status}</div>'
        f'       <div style="margin-bottom: 4px;">{hum_status}</div>'
        f'       <div>{light_status}</div>'
        f'    </div>'
        f'  </div>'
        f'</div>',
        unsafe_allow_html=True
    )

Overwriting app_ui_widgets.py


# Main Application Layout (app.py)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime
import time
import hashlib
import requests

from app_config import apply_custom_styles, get_svg_icon
from app_engine import fetch_latest, fetch_history, fetch_device, DEMO_MODE
from app_ui_widgets import make_combined_chart, render_advisory_card

# Apply Page Setup and Dynamic Canvas Framework Injection
apply_custom_styles()

# ── FIREBASE CREDENTIALS FOR AUTHENTICATION BACKEND ──
FB_URL = "https://smartfarmiot-76187-default-rtdb.firebaseio.com"
FB_AUTH = "LCIE3zkzD3ZSscEhXYGM0os3pDoK51HesZo218xE"

# ── CRYPTOGRAPHIC UTILITY FUNCTIONS ──
def hash_password(password: str) -> str:
    """Applies SHA-256 cryptographic hashing to user credentials."""
    return hashlib.sha256(password.encode()).hexdigest()

def sanitize_email_key(email: str) -> str:
    """Firebase path keys cannot contain periods. Replaces dots to create a valid node identifier."""
    return email.lower().replace(".", "_").replace("@", "_at_")

# ── FIREBASE AUTHENTICATION UTILITIES ──
def firebase_user_exists(email: str) -> bool:
    safe_key = sanitize_email_key(email)
    try:
        response = requests.get(f"{FB_URL}/users/{safe_key}.json?auth={FB_AUTH}")
        return response.json() is not None
    except Exception:
        return False

def firebase_register_user(email: str, password_hash: str) -> bool:
    safe_key = sanitize_email_key(email)
    try:
        user_payload = {
            "email": email.lower(),
            "password": password_hash,
            "created_at": datetime.now().isoformat()
        }
        response = requests.put(f"{FB_URL}/users/{safe_key}.json?auth={FB_AUTH}", json=user_payload)
        return response.status_code == 200
    except Exception:
        return False

def firebase_verify_login(email: str, password_hash: str) -> bool:
    safe_key = sanitize_email_key(email)
    try:
        response = requests.get(f"{FB_URL}/users/{safe_key}.json?auth={FB_AUTH}")
        user_data = response.json()
        if user_data and user_data.get("password") == password_hash:
            return True
        return False
    except Exception:
        return False

# ── INITIALIZE RUNTIME SESSION STATES ──
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "auth_user" not in st.session_state:
    st.session_state.auth_user = None
if "history_hours" not in st.session_state:
    st.session_state.history_hours = 168

# ── SAFE THEME RESOLUTION ──
try:
    is_dark_mode = st.get_option("theme.base") != "light"
except Exception:
    is_dark_mode = False
# ─────────────────────────────────────────────
#  CENTERED AUTHENTICATION PORTAL
# ─────────────────────────────────────────────
if not st.session_state.authenticated:
    # Injecting CSS to tightly center the text inputs and tabs panel
    st.markdown(
        """
        <style>
        /* This forces the background color to flood the absolute root HTML body element */
        html, body, [data-testid="stAppViewContainer"], .stApp {
            background: linear-gradient(135deg, #e8f8ee 0%, #c9f2d5 100%) !important;
            background-attachment: fixed !important;
            min-height: 100vh !important;
            width: 100vw !important;
        }
        /* ── 2. ADD THIS: HIDE THE TRANSPARENT TOP HEADER BAR THAT BLOCKS PLACEMENT ── */
[data-testid="stHeader"] {
    background: transparent !important;
    display: none !important;
}

/* ── 3. ADD THIS: FORCE INDIVIDUAL COLUMNS TO EXPAND FULL WIDTH INSIDE THE MAIN CONTAINER ── */
[data-testid="stColumn"] {
    width: 100% !important;
    flex: 1 1 auto !important;
}

        /* FORCE EVERYTHING inside the auth zone to lock down to a centered layout */
        [data-testid="stHorizontalBlock"] {
            max-width: 550px !important;
            margin: 0 auto !important;
        }

        /* Make all tab panel inputs center perfectly on the screen */
        .stTabs {
            max-width: 550px !important;
            margin: 0 auto !important;
            background: white !important;
            padding: 20px 30px 30px 30px !important;
            border-radius: 18px !important;
            border: 1px solid #e5e7eb !important;
            box-shadow: 0 10px 25px rgba(0,0,0,0.05) !important;
        }

        /* Bold Black text for all widget labels and titles */
        .stTextInput label p, .stTabs button p, label[data-testid="stWidgetLabel"] p {
            color: #000000 !important;
            font-weight: 700 !important;
            font-size: 1rem !important;
        }

        /* Active Tab Header Coloring */
        .stTabs button[aria-selected="true"] {
            color: #15803d !important;
            border-bottom-color: #15803d !important;
        }
        </style>
        """,
        unsafe_allow_html=True
    )

    # Simple column arrangement that will now stay contained securely
    left_space, center_card, right_space = st.columns([0.1, 9.8, 0.1])

    with center_card:
        # 1. Centered Header Card
        st.markdown(
            """
            <div style="background: white; border-radius: 18px; padding: 25px 30px; border: 1px solid #e5e7eb;
                        box-shadow: 0 8px 24px rgba(0,0,0,0.06); max-width: 550px; margin: 40px auto 20px auto;">
                <div style="display: flex; align-items: center; justify-content: center; gap: 18px;">
                    <div style="font-size: 52px;">🌱</div>
                    <div>
                        <h1 style="margin: 0; color: #166534; font-size: 2rem; font-weight: 800; font-family: sans-serif;">
                            ShambaIntel
                        </h1>
                        <p style="margin: 4px 0 0; color: #000000; font-size: 0.95rem; font-weight: 700;">
                            Smart Farm Monitoring System
                        </p>
                    </div>
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

        # 2. Main Authentication Tabs Container (Now locked to a maximum center size)
        auth_tabs = st.tabs([" Sign In ", " Register "])

        with auth_tabs[0]:
            st.markdown("<br>", unsafe_allow_html=True)
            login_email = st.text_input("Registered Email Address", key="login_email_input", placeholder="name@domain.com")
            login_pass = st.text_input("Password", type="password", key="login_pwd")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Sign In", use_container_width=True):
                if not login_email or not login_pass:
                    st.error("Submission failed: Fields cannot be left empty.")
                elif "@" not in login_email or "." not in login_email:
                    st.error("Formatting Error: Please provide a valid email structure.")
                else:
                    hashed = hash_password(login_pass)
                    if firebase_verify_login(login_email, hashed):
                        st.session_state.authenticated = True
                        st.session_state.auth_user = login_email.lower()
                        st.success("Login successful! Welcome to ShambaIntel...")
                        time.sleep(1)
                        st.rerun()
                    else:
                        st.error("Authentication Rejected: Invalid email or encrypted password verification.")

        with auth_tabs[1]:
            st.markdown("<br>", unsafe_allow_html=True)
            reg_email = st.text_input("Email Address", key="reg_email_input", placeholder="farmer@shambaintel.com")
            reg_pass = st.text_input("Create Secure Password", type="password", key="reg_pwd")
            reg_pass_conf = st.text_input("Confirm Secure Password", type="password", key="reg_pwd_conf")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account", use_container_width=True):
                if not reg_email or not reg_pass:
                    st.error("Registration failed: Fields cannot be left empty.")
                elif "@" not in reg_email or "." not in reg_email:
                    st.error("A valid email address is required.")
                elif reg_pass != reg_pass_conf:
                    st.error("Password fields do not match.")
                elif firebase_user_exists(reg_email):
                    st.error("This email is already registered.")
                else:
                    hashed = hash_password(reg_pass)
                    if firebase_register_user(reg_email, hashed):
                        st.success("Account created successfully! You can now sign in.")
                    else:
                        st.error("Unable to create your account. Please try again.")

    st.stop()

# ─────────────────────────────────────────────
#  PRESENTATION TESTING
# Uncomment ONE test line below to present that scenario.


# TEST 1: Test Saturated/Wet Conditions
# latest_override = {"soil_moisture": 95.0, "temperature": 24.5, "humidity": 55.0, "light_level": 350.0, "timestamp": "2026-07-02T12:00:00Z", "alerts": {}}

# TEST 2: Test everything working perfectly (All Green Status)
# latest_override = {"soil_moisture": 65.0, "temperature": 24.5, "humidity": 55.0, "light_level": 500.0, "timestamp": "2026-07-02T12:00:00Z", "alerts": {}}

# TEST 3: Test critically dry soil (Triggers Irrigation Request)
# latest_override = {"soil_moisture": 15.0, "temperature": 32.0, "humidity": 42.0, "light_level": 600.0, "timestamp": "2026-07-02T12:00:00Z", "alerts": {"irrigate": True}}

# TEST 4: Test Severe Environmental Heat & Evaporation Stress
# latest_override = {"soil_moisture": 45.0, "temperature": 39.5, "humidity": 25.0, "light_level": 850.0, "timestamp": "2026-07-02T12:00:00Z", "alerts": {"heat_stress": True, "dry_air": True}}

# TEST 5: Test Low Light / Poor Radiation Conditions
# latest_override = {"soil_moisture": 55.0, "temperature": 21.0, "humidity": 60.0, "light_level": 80.0, "timestamp": "2026-07-02T12:00:00Z", "alerts": {"poor_light": True}}

if 'latest_override' in locals():
    latest = latest_override
    st.sidebar.warning(" Simulation Mode Active")
else:
    latest = fetch_latest()

history = fetch_history(st.session_state.history_hours)
device = fetch_device()
df = pd.DataFrame(history)

if not df.empty:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

try:
    ts_str = datetime.fromisoformat(latest["timestamp"]).astimezone().strftime("%I:%M:%S %p | %b %d, %Y")
except Exception:
    ts_str = "—"

# Global Diagnostics Calculations
bridge_alerts = latest.get("alerts", {}) or {}
is_dry = latest.get("soil_moisture", 0.0) < 30.0 or bridge_alerts.get("irrigate")
is_flooded = latest.get("soil_moisture", 0.0) > 90.0
is_hot = latest.get("temperature", 0.0) > 35.0 or bridge_alerts.get("heat_stress")
is_arid = latest.get("humidity", 0.0) < 40.0 or bridge_alerts.get("dry_air")
is_dark = latest.get("light_level", 0.0) < 200.0 or bridge_alerts.get("poor_light")
any_alerts = is_dry or is_flooded or is_hot or is_arid or is_dark

# ─────────────────────────────────────────────
# VISUAL ASSEMBLY DRAW PIPELINE
# ─────────────────────────────────────────────

# 1. Header & Intro
st.markdown(
    f'<div class="sf-header" style="flex-direction: column; align-items: flex-start; gap: 12px; margin-bottom: 10px;">'
    f'  <div style="display: flex; justify-content: space-between; width: 100%; align-items: center;">'
    f'    <div>'
    f'      <h1 style="margin:0; font-family:\'Space Grotesk\', sans-serif; font-weight:700; letter-spacing:-0.02em;">ShambaIntel Dashboard</h1>'
    f'      <div style="font-size:0.85rem; opacity:0.8; margin-top:2px; font-weight: 500;">Smart Farm Monitoring System</div>'
    f'    </div>'
    f'    <div style="text-align:right; font-family:\'Space Grotesk\', sans-serif; font-size:0.85rem;">'
    f'      <strong>Node User:</strong> <span style="color:#10b981; font-weight:bold;">{st.session_state.auth_user}</span><br>'
    f'      <small style="opacity:0.7;">Sync: {ts_str}</small>'
    f'    </div>'
    f'  </div>'
    f'  <hr style="border: 0; border-top: 1px solid rgba(0,0,0,0.1); width: 100%; margin: 4px 0;">'
    f'<p style="margin: 0; font-size: 0.95rem; line-height: 1.5; color: #000000; font-weight: 500;">'
    f'    Welcome to <b>ShambaIntel</b>. ShambaIntel collects real-time data from ESP32 sensors '
    f'  and monitors soil moisture, temperature, humidity, and light levels to help farmers understand field conditions. '
    f'    The dashboard displays live sensor readings and supports better irrigation decisions to improve crop growth and farm productivity.'
    f'</p>'
    f'</div>',
    unsafe_allow_html=True
)

#  High-Visibility System Alert Banner
if any_alerts:
    alert_messages = []
    if is_dry: alert_messages.append(" Soil is too dry. Water the crops.")
    if is_flooded: alert_messages.append(" Soil is too wet. Reduce watering.")
    if is_hot: alert_messages.append(" Temperature is too high. Crops may be affected.")
    if is_arid: alert_messages.append(" Air is too dry.")
    if is_dark: alert_messages.append(" Light level is too low.")

    combined_banner_text = " | ".join(alert_messages)
    st.markdown(
        f'<div style="background-color: #fecaca; border-left: 8px solid #dc2626; padding: 14px 20px; border-radius: 6px; margin: 10px 0 20px 0;">'
        f'  <span style="color: #991b1b; font-family: \'Space Grotesk\', sans-serif; font-weight: 700; font-size: 1.05rem; display: block;">'
        f'     {combined_banner_text}'
        f'  </span>'
        f'</div>',
        unsafe_allow_html=True
    )

if not any_alerts:
    st.info("**System Check:** All sensors are working normally. Your farm environment looks good!")
else:
    st.warning("**System Check:** Unusual sensor readings detected. Please review the warnings and recommendations below.")
st.markdown("<br>", unsafe_allow_html=True)

# 2. Split Screen Responsive Workspace Layout
left_panel, right_panel = st.columns([1, 2])

with left_panel:
    icon_alert = get_svg_icon("alert", "#f59e0b")
    st.markdown(
        f'<div style="display:flex; align-items:center; gap:8px; margin-bottom:12px;">'
        f'  {icon_alert} <h3 style="margin:0;">Sensor Alerts & Warnings</h3>'
        f'</div>',
        unsafe_allow_html=True
    )

    # Specific Warning Logic
    if latest.get("soil_moisture", 0.0) < 30.0 or bridge_alerts.get("irrigate"):
        st.error("Soil is very dry. Watering is needed.")
    elif latest.get("soil_moisture", 0.0) > 90.0:
        st.error("Flooded Soil Warning. Water levels are excessively high.")

    if latest.get("temperature", 0.0) > 35.0 or bridge_alerts.get("heat_stress"):
        st.error("Temperature is too high. Crops may dry out quickly.")
    if latest.get("humidity", 0.0) < 40.0 or bridge_alerts.get("dry_air"):
        st.warning("Air humidity is low.")
    if latest.get("light_level", 0.0) < 200.0 or bridge_alerts.get("poor_light"):
        st.warning("Not enough sunlight for healthy plant growth.")

    if not any_alerts:
        st.success("All farm conditions are normal.")

    st.markdown("<br>", unsafe_allow_html=True)

    icon_soil = get_svg_icon("soil", "#10b981")
    st.markdown(
        f'<div style="display:flex; align-items:center; gap:8px; margin-bottom:12px;">'
        f'  {icon_soil} <h3 style="margin:0;">Farming Advice & Actions</h3>'
        f'</div>',
        unsafe_allow_html=True
    )
    # Passed all required variables to clean up missing argument exceptions
    render_advisory_card(
        latest.get("soil_moisture", 0.0),
        latest.get("temperature", 0.0),
        latest.get("humidity", 0.0),
        latest.get("light_level", 0.0)
    )

with right_panel:
    icon_dash = get_svg_icon("light", "#6bb343")
    st.markdown(
        f'<div style="display:flex; align-items:center; gap:8px; margin-bottom:12px;">'
        f'  {icon_dash} <h3 style="margin:0;">Live Farm Readings</h3>'
        f'</div>',
        unsafe_allow_html=True
    )

    # Grid metrics distribution
    m1, m2, m3, m4 = st.columns(4)
    with m1: st.metric(label="Soil Moisture", value=f"{latest.get('soil_moisture', 0.0):.1f}%")
    with m2: st.metric(label="Temperature", value=f"{latest.get('temperature', 0.0):.1f}°C")
    with m3: st.metric(label="Air Humidity", value=f"{latest.get('humidity', 0.0):.1f}%")
    with m4: st.metric(label="Light Level", value=f"{latest.get('light_level', 0.0):.1f} lx")

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("### Sensor History Charts")

    time_options = {168: "Lab Session History (Past 1 Week)", 720: "Lab Session History (Past 1 Month)"}
    h_choice = st.selectbox("Select Time Range", options=list(time_options.keys()), index=0, format_func=lambda x: time_options[x])

    if h_choice != st.session_state.history_hours:
        st.session_state.history_hours = h_choice
        st.rerun()

    if df.empty:
        st.info("No sensor records found for this time range.")
    else:
        st.plotly_chart(make_combined_chart(df, is_dark_mode), use_container_width=True)

# 3. Clean Base Footer with Logout Node Switch
st.divider()
f1, f2, f3 = st.columns([2, 1, 1])
with f1:
    if DEMO_MODE: st.info("Status: Running Demo")
    else: st.success("Status: Connected to Live Database")
with f2:
    if st.button("Force Re-evaluation", use_container_width=True): st.rerun()
with f3:
    if st.button("Sign Out", use_container_width=True):
        st.session_state.authenticated = False
        st.session_state.auth_user = None
        st.rerun()

Overwriting app.py


# Clean and Launch

In [ ]:
!streamlit run app.py \
  --server.port 8501 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > streamlit_log.txt 2>&1 &

import time
time.sleep(6)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>